In [ ]:
import pandas as pd
from scipy import stats
from cliffs_delta import cliffs_delta

# 1. ĐỌC DỮ LIỆU TỪ CÁC FILE CSV
# Yêu cầu: Đã chạy xong bước của LR và bước Mutation của Thắng
try:
    df_bc = pd.read_csv('bc_csr.csv')
    df_ms = pd.read_csv('mutation_scores.csv')
    df = pd.merge(df_bc, df_ms, on='func_id')
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file bc_csr.csv hoặc mutation_scores.csv trong thư mục results/.")
    print("Đang sử dụng dữ liệu giả lập (Mock Data) để kiểm tra logic script...\n")
    # Dữ liệu giả lập nếu chưa có file thật
    data = {
        'func_id': [f'PY-{i:03d}' for i in range(1, 51)],
        'bc_gpt': [85.5] * 50, 'bc_pynguin': [70.0] * 50,
        'ms_gpt': [65.0] * 50, 'ms_pynguin': [50.0] * 50,
        'csr_gpt': [1] * 42 + [0] * 8  # 42/50 pass = 84%
    }
    df = pd.DataFrame(data)

# 2. CẤU HÌNH KIỂM ĐỊNH & BONFERRONI CORRECTION
ALPHA = 0.05
NUM_TESTS = 3  # Kiểm định 3 giả thuyết cùng lúc
ALPHA_ADJ = ALPHA / NUM_TESTS

print(f"Ngưỡng Alpha gốc: {ALPHA}")
print(f"Ngưỡng Alpha hiệu chỉnh (Bonferroni): {ALPHA_ADJ:.4f}\n")
print("=" * 60)

# 3. KIỂM ĐỊNH COMPILATION SUCCESS RATE (CSR) - Binomial Exact Test
n_trials = len(df)
n_success = df['csr_gpt'].sum()
csr_rate = n_success / n_trials

binom_res = stats.binomtest(k=int(n_success), n=int(n_trials), p=0.8, alternative='greater')

print("\n[1] KIỂM ĐỊNH COMPILATION SUCCESS RATE (CSR)")
print(f"Tỉ lệ CSR thực tế: {csr_rate*100:.1f}% ({n_success}/{n_trials})")
print(f"P-value: {binom_res.pvalue:.4e}")
if binom_res.pvalue < ALPHA_ADJ:
    print("✅ Kết luận: Bác bỏ H0. CSR của GPT cao hơn 80% một cách có ý nghĩa thống kê.")
else:
    print("❌ Kết luận: Không đủ cơ sở bác bỏ H0. CSR của GPT chưa chắc vượt mức 80%.")

# 4. KIỂM ĐỊNH BRANCH COVERAGE (BC) - Wilcoxon Signed-Rank
wilcoxon_bc = stats.wilcoxon(df['bc_gpt'], df['bc_pynguin'], alternative='greater')
delta_bc, res_bc = cliffs_delta(df['bc_gpt'], df['bc_pynguin'])

print("\n[2] KIỂM ĐỊNH BRANCH COVERAGE (BC)")
print(f"P-value: {wilcoxon_bc.pvalue:.4e}")
print(f"Cliff's Delta: {delta_bc:.4f} (Mức độ: {res_bc})")
if wilcoxon_bc.pvalue < ALPHA_ADJ:
    print("✅ Kết luận: Bác bỏ H0. BC của GPT cao hơn Pynguin một cách có ý nghĩa thống kê.")
else:
    print("❌ Kết luận: Không đủ cơ sở bác bỏ H0. Chưa thấy sự vượt trội của GPT về BC.")

# 5. KIỂM ĐỊNH MUTATION SCORE (MS) - Wilcoxon Signed-Rank
wilcoxon_ms = stats.wilcoxon(df['ms_gpt'], df['ms_pynguin'], alternative='greater')
delta_ms, res_ms = cliffs_delta(df['ms_gpt'], df['ms_pynguin'])

print("\n[3] KIỂM ĐỊNH MUTATION SCORE (MS)")
print(f"P-value: {wilcoxon_ms.pvalue:.4e}")
print(f"Cliff's Delta: {delta_ms:.4f} (Mức độ: {res_ms})")
if wilcoxon_ms.pvalue < ALPHA_ADJ:
    print("✅ Kết luận: Bác bỏ H0. MS của GPT cao hơn Pynguin một cách có ý nghĩa thống kê.")
else:
    print("❌ Kết luận: Không đủ cơ sở bác bỏ H0. Chưa thấy sự vượt trội của GPT về MS.")
print("\n" + "=" * 60)